In [ ]:
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util

# ==========================================
# BƯỚC 0: TẢI DATASET SQuAD TỪ HUGGING FACE
# ==========================================
print("Đang tải dữ liệu từ Hugging Face (SQuAD English)...")
# Tải 100 mẫu đầu tiên của tập validation để demo chạy nhanh
dataset = load_dataset("squad", split="validation[:100]")

# Trích xuất các đoạn văn bản (context) để làm Cơ sở dữ liệu (Corpus)
# Dùng set() để loại bỏ các đoạn văn bản trùng lặp
corpus = list(set(dataset['context']))
print(f" Đã tạo Knowledge Base với {len(corpus)} tài liệu thực tế.\n")

# Lấy một câu hỏi mẫu từ dataset (ở vị trí số 10)
sample_idx = 10
query = dataset['question'][sample_idx]
ground_truth = dataset['context'][sample_idx] # Tài liệu chứa đáp án đúng

print(f"CÂU HỎI TỪ NGƯỜI DÙNG: '{query}'")
print("="*60)


# ==========================================
# 11.1 SPARSE RETRIEVAL (TF-IDF)
# ==========================================
print("\n--- 1. TÌM KIẾM TỪ KHÓA (SPARSE - TF-IDF) ---")
# TF-IDF hoạt động rất tốt với tiếng Anh nhờ khoảng trắng rõ ràng
vectorizer = TfidfVectorizer(stop_words='english')
sparse_doc_vectors = vectorizer.fit_transform(corpus)
sparse_query_vector = vectorizer.transform([query])

sparse_scores = cosine_similarity(sparse_query_vector, sparse_doc_vectors).flatten()
best_sparse_idx = sparse_scores.argmax()

print(f"[Điểm độ tự tin: {sparse_scores[best_sparse_idx]:.4f}]")
print(f"Kết quả trích xuất:\n{corpus[best_sparse_idx][:200]}...")

if corpus[best_sparse_idx] == ground_truth:
    print("-> Đánh giá: TF-IDF TÌM ĐÚNG tài liệu! ")
else:
    print("-> Đánh giá: TF-IDF TÌM SAI! ")


# ==========================================
# 11.3 DENSE RETRIEVAL (Sentence-Transformers)
# ==========================================
print("\n--- 2. TÌM KIẾM NGỮ NGHĨA (DENSE - AI MODEL) ---")
print("(Đang tải mô hình nhúng tiếng Anh all-MiniLM-L6-v2...)")
# all-MiniLM-L6-v2 là model cực nhẹ, được train chuyên cho tiếng Anh
embedder = SentenceTransformer('all-MiniLM-L6-v2')

dense_doc_vectors = embedder.encode(corpus, convert_to_tensor=True)
dense_query_vector = embedder.encode(query, convert_to_tensor=True)

dense_hits = util.semantic_search(dense_query_vector, dense_doc_vectors, top_k=1)[0]
best_dense_idx = dense_hits[0]['corpus_id']
best_dense_score = dense_hits[0]['score']

print(f"[Điểm độ tự tin: {best_dense_score:.4f}]")
print(f"Kết quả trích xuất:\n{corpus[best_dense_idx][:200]}...")

if corpus[best_dense_idx] == ground_truth:
    print("-> Đánh giá: DENSE MODEL TÌM ĐÚNG tài liệu! ")
else:
    print("-> Đánh giá: DENSE MODEL TÌM SAI! ")

Đang tải dữ liệu từ Hugging Face (SQuAD English)...
 Đã tạo Knowledge Base với 4 tài liệu thực tế.

CÂU HỎI TỪ NGƯỜI DÙNG: 'What day was the Super Bowl played on?'

--- 1. TÌM KIẾM TỪ KHÓA (SPARSE - TF-IDF) ---
[Điểm độ tự tin: 0.2881]
Kết quả trích xuất:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated...
-> Đánh giá: TF-IDF TÌM ĐÚNG tài liệu! 

--- 2. TÌM KIẾM NGỮ NGHĨA (DENSE - AI MODEL) ---
(Đang tải mô hình nhúng tiếng Anh all-MiniLM-L6-v2...)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Điểm độ tự tin: 0.6434]
Kết quả trích xuất:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated...
-> Đánh giá: DENSE MODEL TÌM ĐÚNG tài liệu! 


In [ ]:
from huggingface_hub import InferenceClient

print("\n--- 3. RAG THẬT (KẾT NỐI VỚI LLM QUA API CHAT) ---")

# Nhớ thay bằng Token thật của bạn
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxx"

# Khởi tạo client
client = InferenceClient(model="HuggingFaceH4/zephyr-7b-beta", token=HF_TOKEN)

def real_llm_chat(context, user_query):
    """Hàm gọi API chuẩn Chat (Conversational) của Hugging Face"""
    print("(Đang gửi ngữ cảnh cho LLM đọc và suy nghĩ...)")

    # Cách mới: Chia rõ vai trò System (Hệ thống) và User (Người dùng)
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant. Answer the QUESTION based ONLY on the provided CONTEXT. If the context does not contain the answer, say 'I don't know'."
        },
        {
            "role": "user",
            "content": f"CONTEXT:\n{context}\n\nQUESTION: {user_query}"
        }
    ]

    try:
        # Gọi hàm chat_completion thay vì text_generation
        response = client.chat_completion(
            messages=messages,
            max_tokens=100,  # Lưu ý: Tham số này đã đổi tên từ max_new_tokens thành max_tokens
            temperature=0.1
        )
        # Trích xuất nội dung câu trả lời
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Lỗi kết nối API: {e}"

# Lấy tài liệu từ bước Dense Retrieval ở trên
retrieved_context = corpus[best_dense_idx]

# Chạy RAG với API mới
final_answer = real_llm_chat(retrieved_context, query)

print("\n🤖 Trợ lý ảo AI trả lời:")
print(final_answer)


--- 3. RAG THẬT (KẾT NỐI VỚI LLM QUA API CHAT) ---
(Đang gửi ngữ cảnh cho LLM đọc và suy nghĩ...)

🤖 Trợ lý ảo AI trả lời:
Lỗi kết nối API: Client error '401 Unauthorized' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-69e8e981-6db4327d7c04ae9f4e916275;9123d022-798f-4c69-b698-d06c776a89f3)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.
